# Clase 14 — Casos de Estudio: Transformaciones y Acciones

Este notebook desarrolla **únicamente los casos de estudio y sus preguntas** de:

- **Sesión 1: Transformaciones en RDDs** → 🏢 *Caso de Estudio 1: MercaData S.L.*
- **Sesión 2: Acciones en RDDs y Persistencia** → 📞 *Caso de Estudio 2: Atención360 S.L.*

Entorno: Apache Spark 4.1.1 + Scala 2.13 (Almond kernel) en modo `local[*]`.

## 0. Inicialización de Spark

Creamos la `SparkSession`, obtenemos el `SparkContext` y silenciamos los logs para que la salida sea legible.

In [1]:
import $ivy.`org.apache.spark::spark-sql:4.1.1`
import org.apache.log4j.{Level, Logger}
Logger.getLogger("org").setLevel(Level.ERROR)
Logger.getLogger("akka").setLevel(Level.ERROR)

import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Clase14-Casos-Transformaciones-Acciones")
  .master("local[*]")
  .config("spark.ui.showConsoleProgress", "false")
  .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
val sc = spark.sparkContext

println(s"Spark version: ${spark.version}")
println(s"Master:        ${sc.master}")
println(s"App ID:        ${sc.applicationId}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/28 03:48:40 INFO SparkContext: Running Spark version 4.1.1
26/04/28 03:48:40 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/04/28 03:48:40 INFO SparkContext: Java version 17.0.18+8
26/04/28 03:48:41 INFO ResourceUtils: ==============================================================
26/04/28 03:48:41 INFO ResourceUtils: No custom resources configured for spark.driver.
26/04/28 03:48:41 INFO ResourceUtils: ==============================================================
26/04/28 03:48:41 INFO SparkContext: Submitted application: Clase14-Casos-Transformaciones-Acciones
26/04/28 03:48:41 INFO SecurityManager: Changing view acls to: gre
26/04/28 03:48:41 INFO SecurityManager: Changing modify acls to: gre
26/04/28 03:48:41 INFO SecurityManager: Changing view acls groups to: gre
26/04/28 03:48:41 INFO SecurityManager: Changing modify acls groups to: gre
26/04/28 03:48:41 INFO SecurityManager: SecurityMa

Spark version: 4.1.1
Master:        local[*]
App ID:        local-1777340923037


import $ivy.$
import org.apache.log4j.{Level, Logger}
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@61f77a88
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@5c3202e0

---

# 🏢 Caso de Estudio 1 — MercaData S.L.

**Sesión 1: Transformaciones en RDDs**

**MercaData S.L.** es una cadena de supermercados con tiendas en Madrid, Barcelona, Valencia y Sevilla. Han migrado el análisis de ventas a Spark y nos contratan como analistas para responder a 6 preguntas de negocio sobre los datos de la última semana.

**Formato de cada transacción:** `"ID_TIENDA|PROVINCIA|PRODUCTO|CATEGORIA|IMPORTE|EMPLEADO_ID"`

## 📂 Datos de partida

In [2]:
val transacciones = sc.parallelize(List(
  "T01|Madrid|Leche Entera|Lácteos|1.20|E03",
  "T02|Barcelona|Pan de Molde|Panadería|1.85|E07",
  "T03|Valencia|Leche Entera|Lácteos|1.20|E11",
  "T04|Madrid|Zumo de Naranja|Bebidas|2.40|E03",
  "T05|Sevilla|Yogur Natural|Lácteos|0.75|E15",
  "T06|Barcelona|Agua Mineral|Bebidas|0.60|E07",
  "T07|Madrid|Cerveza Rubia|Bebidas|1.10|E04",
  "T08|Valencia|Pan de Molde|Panadería|1.85|E12",
  "T09|Sevilla|Leche Entera|Lácteos|1.20|E15",
  "T10|Madrid|Yogur Natural|Lácteos|0.75|E03",
  "T11|Barcelona|Zumo de Naranja|Bebidas|2.40|E08",
  "T12|Valencia|Cerveza Rubia|Bebidas|1.10|E11",
  "T13|Sevilla|Agua Mineral|Bebidas|0.60|E16",
  "T14|Madrid|Leche Entera|Lácteos|1.20|E04",
  "T15|Barcelona|Pan de Molde|Panadería|1.85|E07",
  "T16|Valencia|Yogur Natural|Lácteos|0.75|E12",
  "T17|Sevilla|Zumo de Naranja|Bebidas|2.40|E15",
  "T18|Madrid|Agua Mineral|Bebidas|0.60|E03",
  "T19|Barcelona|Leche Entera|Lácteos|1.20|E08",
  "T20|Valencia|Pan de Molde|Panadería|1.85|E11",
  "T21|Sevilla|Cerveza Rubia|Bebidas|1.10|E16",
  "T22|Madrid|Zumo de Naranja|Bebidas|2.40|E04",
  "T23|Barcelona|Yogur Natural|Lácteos|0.75|E07",
  "T24|Valencia|Leche Entera|Lácteos|1.20|E12",
  "T25|Sevilla|Pan de Molde|Panadería|1.85|E15"
))

val empleados = sc.parallelize(List(
  ("E03", "Carmen Vidal"),
  ("E04", "Luis Herrero"),
  ("E07", "Marta Soler"),
  ("E08", "Diego Fuentes"),
  ("E11", "Ana Romero"),
  ("E12", "Pablo Leal"),
  ("E15", "Rosa Cano"),
  ("E16", "Javier Mora")
))

println(s"Transacciones cargadas: ${transacciones.count()}")
println(s"Empleados cargados:     ${empleados.count()}")

Transacciones cargadas: 25
Empleados cargados:     8


transacciones: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:1
empleados: org.apache.spark.rdd.RDD[(String, String)] = ParallelCollectionRDD[1] at parallelize at cmd2.sc:29

## ❓ Pregunta 1 — ¿Cuánto ha facturado cada provincia?

Importe total vendido en cada provincia, ordenado de mayor a menor.

In [3]:
val facturacionPorProvincia = transacciones
  .map { linea =>
    val campos = linea.split("\\|")
    (campos(1), campos(4).toDouble)   // (provincia, importe)
  }
  .reduceByKey(_ + _)
  .sortBy(_._2, ascending = false)

println("Facturación por provincia (mayor a menor):")
facturacionPorProvincia.collect().zipWithIndex.foreach { case ((prov, total), i) =>
  println(f"  ${i + 1}. ${prov}%-10s → ${total}%.2f€")
}

Facturación por provincia (mayor a menor):
  1. Madrid     → 9,65€
  2. Barcelona  → 8,65€
  3. Valencia   → 7,95€
  4. Sevilla    → 7,90€


facturacionPorProvincia: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[8] at sortBy at cmd3.sc:7

## ❓ Pregunta 2 — ¿Qué categorías de producto se venden en Madrid?

Categorías presentes en Madrid, **sin repetidos**.

In [4]:
val categoriasMadrid = transacciones
  .filter(_.split("\\|")(1) == "Madrid")
  .map(_.split("\\|")(3))     // categoría
  .distinct()

println("Categorías en Madrid:")
categoriasMadrid.collect().sorted.foreach(c => println(s"  $c"))

Categorías en Madrid:
  Bebidas
  Lácteos


categoriasMadrid: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[13] at distinct at cmd4.sc:3

## ❓ Pregunta 3 — ¿Cuántas transacciones ha gestionado cada empleado?

Mostramos el **nombre completo** sustituyendo el ID mediante un `join` con el RDD `empleados`.

In [5]:
// Paso 1: contar transacciones por ID de empleado
val conteoPorId = transacciones
  .map(linea => (linea.split("\\|")(5), 1))   // (id, 1)
  .reduceByKey(_ + _)

// Paso 2: cruzar con empleados → (id, (conteo, nombre))
val conteoPorEmpleado = conteoPorId
  .join(empleados)
  .map { case (id, (n, nombre)) => (nombre, n) }
  .sortByKey()

println("Transacciones por empleado:")
conteoPorEmpleado.collect().foreach { case (nombre, n) =>
  println(f"  ${nombre}%-15s → $n transacciones")
}

Transacciones por empleado:
  Ana Romero      → 3 transacciones
  Carmen Vidal    → 4 transacciones
  Diego Fuentes   → 2 transacciones
  Javier Mora     → 2 transacciones
  Luis Herrero    → 3 transacciones
  Marta Soler     → 4 transacciones
  Pablo Leal      → 3 transacciones
  Rosa Cano       → 4 transacciones


conteoPorId: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[15] at reduceByKey at cmd5.sc:4
conteoPorEmpleado: org.apache.spark.rdd.RDD[(String, Int)] = ShuffledRDD[22] at sortByKey at cmd5.sc:10

## ❓ Pregunta 4 — ¿Qué productos se venden tanto en Madrid como en Barcelona?

In [6]:
val productosMadrid = transacciones
  .filter(_.split("\\|")(1) == "Madrid")
  .map(_.split("\\|")(2))

val productosBarcelona = transacciones
  .filter(_.split("\\|")(1) == "Barcelona")
  .map(_.split("\\|")(2))

val comunes = productosMadrid.intersection(productosBarcelona)

println("Productos en Madrid Y Barcelona:")
comunes.collect().sorted.foreach(p => println(s"  $p"))

Productos en Madrid Y Barcelona:
  Agua Mineral
  Leche Entera
  Yogur Natural
  Zumo de Naranja


productosMadrid: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[24] at map at cmd6.sc:3
productosBarcelona: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[26] at map at cmd6.sc:7
comunes: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[32] at intersection at cmd6.sc:9

## ❓ Pregunta 5 — Facturación total por categoría

In [7]:
val facturacionPorCategoria = transacciones
  .map { linea =>
    val c = linea.split("\\|")
    (c(3), c(4).toDouble)     // (categoría, importe)
  }
  .reduceByKey(_ + _)
  .sortByKey()

println("Facturación por categoría:")
facturacionPorCategoria.collect().foreach { case (cat, total) =>
  println(f"  ${cat}%-10s → ${total}%.2f€")
}

Facturación por categoría:
  Bebidas    → 14,70€
  Lácteos    → 10,20€
  Panadería  → 9,25€


facturacionPorCategoria: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[37] at sortByKey at cmd7.sc:7

## ❓ Pregunta 6 — Catálogo completo de productos únicos

In [8]:
val catalogo = transacciones
  .map(_.split("\\|")(2))    // producto
  .distinct()

println("Catálogo de productos:")
catalogo.collect().sorted.foreach(p => println(s"  $p"))

Catálogo de productos:
  Agua Mineral
  Cerveza Rubia
  Leche Entera
  Pan de Molde
  Yogur Natural
  Zumo de Naranja


catalogo: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[41] at distinct at cmd8.sc:2

---

# 📞 Caso de Estudio 2 — Atención360 S.L.

**Sesión 2: Acciones en RDDs y Persistencia**

**Atención360 S.L.** es una empresa de atención al cliente. Necesitan un informe semanal con métricas de rendimiento (duración, satisfacción y tasa de resolución por agente) usando exclusivamente **acciones RDD**.

**Formato:** `"AGENTE_ID|NOMBRE_AGENTE|DURACION_SEG|SATISFACCION|ESTADO"`

## 📂 Datos y caché

Como vamos a hacer múltiples acciones sobre el mismo RDD, lo persistimos con `cache()`.

In [9]:
val registros = sc.parallelize(List(
  "A01|Laura Méndez|245|4|RESUELTA",
  "A02|Carlos Reyes|180|5|RESUELTA",
  "A01|Laura Méndez|320|3|NO_RESUELTA",
  "A03|Sofía Ibáñez|95|5|RESUELTA",
  "A02|Carlos Reyes|410|2|NO_RESUELTA",
  "A01|Laura Méndez|150|5|RESUELTA",
  "A03|Sofía Ibáñez|280|4|RESUELTA",
  "A02|Carlos Reyes|190|4|RESUELTA",
  "A01|Laura Méndez|530|1|NO_RESUELTA",
  "A03|Sofía Ibáñez|210|5|RESUELTA",
  "A02|Carlos Reyes|175|5|RESUELTA",
  "A01|Laura Méndez|90|4|RESUELTA",
  "A03|Sofía Ibáñez|340|3|NO_RESUELTA",
  "A02|Carlos Reyes|265|4|RESUELTA",
  "A03|Sofía Ibáñez|120|5|RESUELTA",
  "A01|Laura Méndez|480|2|NO_RESUELTA",
  "A02|Carlos Reyes|155|5|RESUELTA",
  "A03|Sofía Ibáñez|390|3|NO_RESUELTA",
  "A01|Laura Méndez|220|4|RESUELTA",
  "A02|Carlos Reyes|310|3|RESUELTA"
))

registros.cache()
println(s"RDD cacheado. Total de registros: ${registros.count()}")

RDD cacheado. Total de registros: 20


registros: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[42] at parallelize at cmd9.sc:1
res9_1: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[42] at parallelize at cmd9.sc:1

## ❓ Pregunta 1 — Inspección inicial

**Acciones:** `count`, `first`, `take`.

In [10]:
println(s"Total de llamadas: ${registros.count()}")
println(s"Primer registro: ${registros.first()}")
println("Primeras 3:")
registros.take(3).foreach(r => println(s"  $r"))

Total de llamadas: 20
Primer registro: A01|Laura Méndez|245|4|RESUELTA
Primeras 3:
  A01|Laura Méndez|245|4|RESUELTA
  A02|Carlos Reyes|180|5|RESUELTA
  A01|Laura Méndez|320|3|NO_RESUELTA


## ❓ Pregunta 2 — Estadísticas de duración

**Acción:** `reduce` (tres veces sobre el mismo RDD de duraciones).

In [11]:
val duraciones = registros.map(_.split("\\|")(2).toInt)

val duracionTotal  = duraciones.reduce(_ + _)
val duracionMaxima = duraciones.reduce((a, b) => if (a > b) a else b)
val duracionMinima = duraciones.reduce((a, b) => if (a < b) a else b)

println(s"Duración total:   $duracionTotal segundos")
println(s"Duración máxima:  $duracionMaxima segundos")
println(s"Duración mínima:  $duracionMinima segundos")

Duración total:   5155 segundos
Duración máxima:  530 segundos
Duración mínima:  90 segundos


duraciones: org.apache.spark.rdd.RDD[Int] = MapPartitionsRDD[43] at map at cmd11.sc:1
duracionTotal: Int = 5155
duracionMaxima: Int = 530
duracionMinima: Int = 90

## ❓ Pregunta 3 — Tasa de resolución global

**Acción:** `filter` + `count`.

In [12]:
val total       = registros.count()
val resueltas   = registros.filter(_.split("\\|")(4) == "RESUELTA").count()
val noResueltas = total - resueltas
val tasa        = (resueltas.toDouble / total) * 100

println(s"Llamadas resueltas:     $resueltas")
println(s"Llamadas no resueltas:  $noResueltas")
println(f"Tasa de resolución:     $tasa%.2f%%")

Llamadas resueltas:     14
Llamadas no resueltas:  6
Tasa de resolución:     70,00%


total: Long = 20L
resueltas: Long = 14L
noResueltas: Long = 6L
tasa: Double = 70.0

## ❓ Pregunta 4 — Media de satisfacción y duración en una sola pasada

**Acción:** `aggregate` con acumulador `(sumaDuracion, sumaSatisfaccion, conteo)`.

In [13]:
// Acumulador: (sumaDuracion, sumaSatisfaccion, conteo)
val (sumDur, sumSat, n) = registros.aggregate((0, 0, 0))(
  (acc, linea) => {
    val c = linea.split("\\|")
    (acc._1 + c(2).toInt, acc._2 + c(3).toInt, acc._3 + 1)
  },
  (a, b) => (a._1 + b._1, a._2 + b._2, a._3 + b._3)
)

val mediaDuracion     = sumDur.toDouble / n
val mediaSatisfaccion = sumSat.toDouble / n

println(f"Media de duración:      $mediaDuracion%.1f segundos")
println(f"Media de satisfacción:  $mediaSatisfaccion%.2f / 5")

Media de duración:      257,8 segundos
Media de satisfacción:  3,80 / 5


sumDur: Int = 5155
sumSat: Int = 76
n: Int = 20
mediaDuracion: Double = 257.75
mediaSatisfaccion: Double = 3.8

## ❓ Pregunta 5 — La llamada más larga

**Acción:** `reduce` directamente sobre las líneas, comparando por la duración.

In [14]:
val masLarga = registros.reduce { (a, b) =>
  val durA = a.split("\\|")(2).toInt
  val durB = b.split("\\|")(2).toInt
  if (durA > durB) a else b
}

val campos = masLarga.split("\\|")
println("Llamada de mayor duración:")
println(s"  Agente:        ${campos(1)} (${campos(0)})")
println(s"  Duración:      ${campos(2)} segundos")
println(s"  Satisfacción:  ${campos(3)} / 5")
println(s"  Estado:        ${campos(4)}")

Llamada de mayor duración:
  Agente:        Laura Méndez (A01)
  Duración:      530 segundos
  Satisfacción:  1 / 5
  Estado:        NO_RESUELTA


masLarga: String = "A01|Laura Méndez|530|1|NO_RESUELTA"
campos: Array[String] = Array("A01", "Laura Méndez", "530", "1", "NO_RESUELTA")

## ❓ Pregunta 6 — Informe por agente y guardado en disco

**Acciones:** `filter` + `aggregate` por agente, `foreach` para imprimir, `saveAsTextFile` para guardar.

Por cada agente calculamos: nº de llamadas, duración media, satisfacción media y nº de resueltas.

In [15]:
// Identificamos los agentes únicos (lo recogemos al driver porque son pocos)
val agentes = registros
  .map { linea =>
    val c = linea.split("\\|")
    (c(0), c(1))     // (id, nombre)
  }
  .distinct()
  .collect()
  .sortBy(_._1)

// Para cada agente: filter + aggregate → línea de informe
val lineasInforme: Array[String] = agentes.map { case (id, nombre) =>
  val rddAgente = registros.filter(_.split("\\|")(0) == id)

  // Acumulador: (sumDur, sumSat, conteo, resueltas)
  val (sd, ss, cnt, res) = rddAgente.aggregate((0, 0, 0, 0))(
    (acc, linea) => {
      val c = linea.split("\\|")
      val resuelta = if (c(4) == "RESUELTA") 1 else 0
      (acc._1 + c(2).toInt, acc._2 + c(3).toInt, acc._3 + 1, acc._4 + resuelta)
    },
    (a, b) => (a._1 + b._1, a._2 + b._2, a._3 + b._3, a._4 + b._4)
  )

  val durMedia = sd.toDouble / cnt
  val satMedia = ss.toDouble / cnt
  f"  ${nombre}%-13s ($id): $cnt llamadas | dur. media: ${durMedia.toInt}s | sat. media: $satMedia%.2f | resueltas: $res/$cnt"
}

// Imprimir el informe en pantalla
println("========= INFORME SEMANAL DE AGENTES =========")
sc.parallelize(lineasInforme).foreach(println)
// Pequeña espera visual: foreach se ejecuta en executors; en local los println salen igual
lineasInforme.foreach(println)
println("==============================================")

1 deprecation (since 2.13.0); re-run enabling -deprecation for details, or try -help


========= INFORME SEMANAL DE AGENTES =========
  Carlos Reyes  (A02): 7 llamadas | dur. media: 240s | sat. media: 4,00 | resueltas: 6/7
  Sofía Ibáñez  (A03): 6 llamadas | dur. media: 239s | sat. media: 4,17 | resueltas: 4/6
  Laura Méndez  (A01): 7 llamadas | dur. media: 290s | sat. media: 3,29 | resueltas: 4/7
  Laura Méndez  (A01): 7 llamadas | dur. media: 290s | sat. media: 3,29 | resueltas: 4/7
  Carlos Reyes  (A02): 7 llamadas | dur. media: 240s | sat. media: 4,00 | resueltas: 6/7
  Sofía Ibáñez  (A03): 6 llamadas | dur. media: 239s | sat. media: 4,17 | resueltas: 4/6


agentes: Array[(String, String)] = Array(
  ("A01", "Laura Méndez"),
  ("A02", "Carlos Reyes"),
  ("A03", "Sofía Ibáñez")
)
lineasInforme: Array[String] = Array(
  "  Laura Méndez  (A01): 7 llamadas | dur. media: 290s | sat. media: 3,29 | resueltas: 4/7",
  "  Carlos Reyes  (A02): 7 llamadas | dur. media: 240s | sat. media: 4,00 | resueltas: 6/7",
  "  Sofía Ibáñez  (A03): 6 llamadas | dur. media: 239s | sat. media: 4,17 | resueltas: 4/6"
)

In [16]:
// Guardar el informe en disco con saveAsTextFile
import org.apache.hadoop.fs.{FileSystem, Path}

val rutaInforme = "C:/Curso-Scala/salida/informe_agentes"
val fs = FileSystem.get(sc.hadoopConfiguration)
if (fs.exists(new Path(rutaInforme))) fs.delete(new Path(rutaInforme), true)

// Construimos un RDD con las líneas del informe y lo guardamos
val cabecera = "========= INFORME SEMANAL DE AGENTES ========="
val cierre   = "=============================================="
val informeRdd = sc.parallelize(cabecera +: lineasInforme :+ cierre, 1)

informeRdd.saveAsTextFile(rutaInforme)
println(s"✅ Informe guardado en: $rutaInforme")

1 deprecation (since 2.13.0); re-run enabling -deprecation for details, or try -help


✅ Informe guardado en: C:/Curso-Scala/salida/informe_agentes


import org.apache.hadoop.fs.{FileSystem, Path}
rutaInforme: String = "C:/Curso-Scala/salida/informe_agentes"
fs: FileSystem = org.apache.hadoop.fs.LocalFileSystem@450a3c3f
res16_3: AnyVal = ()
cabecera: String = "========= INFORME SEMANAL DE AGENTES ========="
cierre: String = "=============================================="
informeRdd: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[53] at parallelize at cmd16.sc:11

In [17]:
// Verificación: leer el fichero recién guardado
val verif = sc.textFile("C:/Curso-Scala/salida/informe_agentes")
println(s"Líneas escritas en disco: ${verif.count()}")
println("Contenido del informe guardado:")
verif.collect().foreach(println)

Líneas escritas en disco: 5
Contenido del informe guardado:
========= INFORME SEMANAL DE AGENTES =========
  Laura Méndez  (A01): 7 llamadas | dur. media: 290s | sat. media: 3,29 | resueltas: 4/7
  Carlos Reyes  (A02): 7 llamadas | dur. media: 240s | sat. media: 4,00 | resueltas: 6/7
  Sofía Ibáñez  (A03): 6 llamadas | dur. media: 239s | sat. media: 4,17 | resueltas: 4/6


verif: org.apache.spark.rdd.RDD[String] = C:/Curso-Scala/salida/informe_agentes MapPartitionsRDD[56] at textFile at cmd17.sc:2

## 🧹 Liberar caché

Buena práctica al terminar el caso de estudio: liberar la memoria reservada por `cache()`.

In [18]:
registros.unpersist()
println("✅ Caché de 'registros' liberada.")

✅ Caché de 'registros' liberada.


res18_0: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[42] at parallelize at cmd9.sc:1

---

## 🛑 Cerrar la sesión de Spark (opcional)

Descomenta la siguiente celda únicamente cuando quieras finalizar todo el trabajo en Spark.

In [18]:
// spark.stop()